conda install pytorch torchvision torchaudio pytorch-cuda=12.4 -c pytorch -c nvidia

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# 간단 실행 테스트
x = torch.randn(3, 3).to("cuda" if torch.cuda.is_available() else "cpu")
print("tensor device:", x.device)

# **CNN-BiLSTM** 

---

## 1. 전체 파이프라인

1) 데이터 로드 및 전처리  
- `train.csv`, `val.csv`, `test.csv` 로드  
- 시간 특성 생성: `hour`, `doy` → `sin`, `cos`  
- 단기 지연값과 차분 생성: `lag_{1,3,6,24}`, `diff_{1,3,6,24}`  
- 결측은 0으로 채움

2) 지역별 스케일링  
- 입력 특징: `StandardScaler → MinMaxScaler`  
- 타깃(`합산발전량(MWh)`): `log1p` 적용 후 `StandardScaler`  
- 학습 데이터에서 지역별 스케일러 적합 후, `train/val/test`에 동일 규칙 적용

3) 데이터셋 구성  
- 슬라이딩 윈도우: `SEQ_LEN=168`, `HORIZON=24`  
- 윈도우 수 = `len(df) - seq_len - horizon + 1`  
- 배치 단위 텐서 반환

4) 모델 학습(Plant 1차)  
- 모델: 1D CNN → BiLSTM → FC  
- 에포크별 학습/검증 → EMA 기반 R² 추적, 조기 종료(`PATIENCE`)  
- 체크포인트 저장: `*_last.pt`, `*_best.pt`  
- 재시작 옵션: `RESUME=True`일 때 last/best에서 이어서 학습

5) 지역 단위 Fine-tuning(2차)  
- 동일 모델로 지역 데이터 학습  
- 같은 지역의 Plant 체크포인트들을 파라미터 평균으로 초기화(존재 시)  
- 검증 성능 최고 시 Best 갱신

6) Permutation Importance 분석(지역 단위)  
- 지역별 Best 체크포인트 로드 후, 테스트 윈도우 일부 샘플링  
- 기상 특징(Weather) 차원만 무작위 순열하여 `RMSE 증가량` 평가  
- 상위 3개 기상 요인 추출 및 수평 바차트로 시각화

7) 이상치 탐지 및 시각화(지역·전체)  
- 테스트 윈도우로 추론 → 스케일 역변환(log1p/표준화 해제)  
- `MWh` 단위 보정(코드상 24시간 누적 처리 포함)  
- 절대 오차의 99% 분위수를 임계값으로 사용하여 이상치 판정  
- 전체/지역별 예측 vs 실제 라인 플롯에 이상치 포인트 오버레이

8) 리포트 생성  
- Plant/Region 성능 표, PI 상위 3개 요인, 예측-실제 그래프, 이상치 집계  
- HTML 파일로 저장 및 자동 오픈

---

## 2. 단계별 주요 함수

| 구분 | 함수/클래스 | 핵심 역할 |
|---|---|---|
| 전처리 | `add_time_feats(df)` | `hour/doy` 계산 및 `sin/cos` 파생 |
|  | `add_lag_diff(df, lags)` | `lag_k`와 `diff_k`(현재−lag_k) 생성, 결측 0 채움 |
| 스케일 | 지역별 루프 | 입력: `StandardScaler → MinMaxScaler`, 타깃: `log1p → StandardScaler` 적합/적용 |
| 데이터셋 | `TimeSeriesSeqDataset` | 슬라이딩 윈도우로 `(X(seq_len×feat), y(horizon))` 생성 |
| 모델 | `CNN_BiLSTM` | Conv1d(입력→128) → Conv1d(128→64) → BiLSTM(64→hidden, 2층) → `LayerNorm` → `FC(horizon)` |
| 학습/평가 | `calc_metrics(y_true, y_pred)` | R²(0 아래 절단), RMSE, MAE 계산 |
|  | `ema_update(prev, curr, alpha)` | EMA로 검증 R² 완화 추적 |
|  | `train_plant(...)` | 1차(Plant) 학습 루프, EMA·조기 종료·ckpt 저장/재개 포함 |
| 체크포인트 | `save_checkpoint(path, ...)` | 모델/옵티마/에포크/지표 저장 |
|  | `load_checkpoint(model, optimizer, path)` | 저장 상태 로드 및 재개 |
| Fine-tune | 지역 루프(메인 코드) | 지역별 데이터 학습, Plant ckpt 평균 초기화, Best 갱신 |
| 중요도 | `analyze_permutation_importance(region, ckpt, test_df)` | 지역별 테스트 샘플로 Weather feature의 순열 중요도 계산 및 상위 3개 시각화 |
| 이상치 | `detect_outliers(region_data, ...)` | 지역/전체 예측-실제 라인 그래프 생성, 99% 분위수 기반 이상치 판정(개수/비율/임계값) |

---

## 3. 데이터·모델·하이퍼파라미터

- 입력 특징  
  `WEATHER_COLS` 10개 + `TIME_FEATS` 4개 + `lag_{1,3,6,24}` + `diff_{1,3,6,24}`  
  실제 사용은 컬럼 존재 여부로 필터링된 `feature_cols`

- 윈도우 및 배치  
  `SEQ_LEN=168`, `HORIZON=24`, `BATCH=128`

- 학습 스케줄  
  1차(Plant): `EPOCHS=50`, 2차(Region): `FINE_TUNE_EPOCHS=10`, `LR=1e-3`  
  조기 종료: `PATIENCE=10`, EMA 계수: `alpha=0.3`(기본 인자)

- 디바이스  
  `DEVICE = cuda` 가능 시 GPU, 그 외 CPU

- 체크포인트 경로  
  `CKPT_DIR/plant/{plant}_best.pt`, `CKPT_DIR/region/{region}_best.pt` 등

---

## 4. 산출물 요약

| 항목 | 내용 |
|---|---|
| 1차 결과 | Plant별 Best 체크포인트(.pt), Plant별 검증 성능표, 가중 평균(R², RMSE, MAE) |
| 2차 결과 | Region별 Best 체크포인트(.pt), Region별 검증 성능표, 가중 평균(R², RMSE, MAE) |
| 중요도 | 지역별 Permutation Importance 바차트(Top3 기상 요인) |
| 이상치 | 전체 및 지역별 예측-실제 라인 그래프 + 이상치 포인트, 이상치 개수/비율/임계값 |
| 리포트 | `plant_region_report_perm.html` (표·그래프·중요도·이상치 요약 포함) |

---

## 5. 처리 흐름(요약)

1) CSV 로드 → `add_time_feats` → `add_lag_diff`  
2) 지역별 스케일러 적합 및 `train/val/test`에 적용  
3) `TimeSeriesSeqDataset`으로 슬라이싱하여 1차(Plant) 학습 → EMA·조기 종료·ckpt 저장  
4) 지역 단위로 Fine-tuning(Plant 평균 ckpt로 초기화 가능) → Best 저장  
5) 지역별 Best 모델로 Permutation Importance 산출(기상 특징만)  
6) 테스트 구간 예측-실제 비교, 99% 분위수 임계값으로 이상치 판정 및 시각화  
7) 성능 표·PI·이상치 결과를 HTML로 종합 출력


In [16]:
# -*- coding: utf-8 -*-
"""
🌱 완전 통합 파이프라인 (Plant → Region)
- CNN-BiLSTM 기반
- 1차 학습 / 조건부 2차 학습 (weight step)
- Region별 Fine-tuning
- Resume, AMP, EarlyStopping
- Permutation Importance 안전 분석 (안전하게 시퀀스 처리)
- HTML/CSV 보고서 자동 생성 + Plant vs Region 성능 비교
- 출력 강화: ETA, Best 표시, 상세 지표
- 음수 R² -> 0 보정 및 제외된 발전소 별도 표기
"""
import os, time, warnings, base64, webbrowser, traceback
import io
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from jinja2 import Template
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {DEVICE}")

# ----------------------------
# === 사용자 설정 ===
# ----------------------------
TRAIN_CSV = r"C:\ESG_Project1\file\merge_data\train.csv"
VAL_CSV   = r"C:\ESG_Project1\file\merge_data\val.csv"
TEST_CSV  = r"C:\ESG_Project1\file\merge_data\test.csv"
SAVE_DIR  = r"C:\ESG_Project1\cnn_bilstm\output"
CKPT_DIR  = os.path.join(SAVE_DIR,"checkpoints")
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

TIME_COL, GROUP_COL, REGION_COL, TARGET_COL = "일시", "발전구분", "지역", "합산발전량(MWh)"
WEATHER_COLS = [
    "기온(°C)", "강수량(mm)", "풍속(m/s)", "습도(%)", "증기압(hPa)",
    "일조(hr)", "일사(MJ/m2)", "적설(cm)", "전운량(10분위)", "중하층운량(10분위)"
]
TIME_FEATS = ["hour_sin", "hour_cos", "doy_sin", "doy_cos"]

SEQ_LEN, HORIZON = 168, 24
BATCH = 128
EPOCHS = 50
FINE_TUNE_EPOCHS = 10
LR = 1e-3
PATIENCE = 10
PERM_SAMPLE = 50
RESUME = True   # Resume 기능 사용 여부 (전체 파이프라인 재개 가능)

# ✅ 한글 폰트 설정 (Windows 기준)
plt.rcParams['font.family'] = 'Malgun Gothic'   # 맑은 고딕
plt.rcParams['axes.unicode_minus'] = False      # 마이너스 깨짐 방지

# ----------------------------
# === 데이터 로드 + 전처리 ===
# ----------------------------
def add_time_feats(df):
    df = df.copy()
    df.sort_values([GROUP_COL, TIME_COL], inplace=True)
    df["hour"] = df[TIME_COL].dt.hour
    df["doy"] = df[TIME_COL].dt.dayofyear
    df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)
    df["doy_sin"] = np.sin(2*np.pi*df["doy"]/365)
    df["doy_cos"] = np.cos(2*np.pi*df["doy"]/365)
    return df

def add_lag_diff(df, lags=[1,3,6,24]):
    df = df.copy()
    for lag in lags:
        df[f"lag_{lag}"] = df.groupby(GROUP_COL)[TARGET_COL].shift(lag)
        df[f"diff_{lag}"] = df[TARGET_COL] - df[f"lag_{lag}"]
    df.fillna(0, inplace=True)
    return df

# load
train_raw = add_lag_diff(add_time_feats(pd.read_csv(TRAIN_CSV, parse_dates=[TIME_COL])))
val_raw   = add_lag_diff(add_time_feats(pd.read_csv(VAL_CSV, parse_dates=[TIME_COL])))
test_raw  = add_lag_diff(add_time_feats(pd.read_csv(TEST_CSV, parse_dates=[TIME_COL])))

all_candidate_feats = WEATHER_COLS + TIME_FEATS + [f"lag_{l}" for l in [1,3,6,24]] + [f"diff_{l}" for l in [1,3,6,24]]
feature_cols = [c for c in all_candidate_feats if c in train_raw.columns]
print(f"✅ feature_cols ({len(feature_cols)}): {feature_cols}")

# ----------------------------
# === 지역별 스케일러 적용 ===
# ----------------------------
region_scalers = {}
for region, grp in train_raw.groupby(REGION_COL):
    valid_cols = [c for c in feature_cols if c in grp.columns]
    if len(valid_cols)==0: continue
    std = StandardScaler().fit(grp[valid_cols])
    mm  = MinMaxScaler().fit(std.transform(grp[valid_cols]))
    tsc = StandardScaler().fit(np.log1p(grp[[TARGET_COL]].clip(lower=0.0)+1e-8))
    region_scalers[region] = (std, mm, tsc)
    train_raw.loc[grp.index, valid_cols] = mm.transform(std.transform(grp[valid_cols]))
    train_raw.loc[grp.index, TARGET_COL] = tsc.transform(np.log1p(grp[[TARGET_COL]].clip(lower=0.0)+1e-8))

for df in (val_raw, test_raw):
    for region, grp in df.groupby(REGION_COL):
        valid_cols = [c for c in feature_cols if c in grp.columns]
        if region in region_scalers and len(valid_cols)>0:
            std, mm, tsc = region_scalers[region]
            df.loc[grp.index, valid_cols] = mm.transform(std.transform(grp[valid_cols]))
            df.loc[grp.index, TARGET_COL] = tsc.transform(np.log1p(grp[[TARGET_COL]].clip(lower=0.0)+1e-8))

# ----------------------------
# === Dataset & Model ===
# ----------------------------
class TimeSeriesSeqDataset(Dataset):
    def __init__(self, df, seq_len=SEQ_LEN, horizon=HORIZON, feature_cols=feature_cols):
        self.seq_len = seq_len
        self.horizon = horizon
        self.features = df[feature_cols].values.astype(np.float32)
        self.targets = df[TARGET_COL].values.astype(np.float32)
        self.n_windows = max(0, len(df) - seq_len - horizon + 1)

    def __len__(self):
        return self.n_windows

    def __getitem__(self, i):
        x = self.features[i:i + self.seq_len]
        # 단일 시점 예측이면 y도 스칼라로 반환
        if self.horizon == 1:
            y = self.targets[i + self.seq_len]
        else:
            y = self.targets[i + self.seq_len : i + self.seq_len + self.horizon]
        return torch.from_numpy(x), torch.from_numpy(np.atleast_1d(y))


class CNN_BiLSTM(nn.Module):
    def __init__(self, input_dim=len(feature_cols), hidden=128, num_layers=2, horizon=HORIZON):
        super().__init__()
        self.conv1 = nn.Conv1d(input_dim, 128, 3, padding=1)
        self.conv2 = nn.Conv1d(128, 64, 3, padding=1)
        self.lstm = nn.LSTM(64, hidden, num_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden*2, horizon)
        self.relu = nn.ReLU()
        self.norm = nn.LayerNorm(hidden*2)
    def forward(self, x):
        x = x.permute(0,2,1)            # (B, C, T)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = x.permute(0,2,1)            # (B, T, C)
        x,_ = self.lstm(x)
        x = self.norm(x[:,-1,:])
        return self.fc(x)

# =========================
# === Utility Functions ===
# =========================
def calc_metrics(y_true, y_pred):
    if len(y_true)==0: return 0.0, 0.0, 0.0
    r2 = r2_score(y_true, y_pred)
    r2 = max(r2,0.0)  # 음수 R² 보정
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    return r2, rmse, mae

def save_checkpoint(path, model, optimizer, epoch, metric, extra=None):
    ckpt = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "metric": metric
    }
    if extra: ckpt.update(extra)
    torch.save(ckpt, path)

def load_checkpoint(model, optimizer, path):
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    return ckpt["epoch"], ckpt.get("metric",0.0)

def ema_update(prev, curr, alpha=0.3):
    if prev is None: return curr
    return alpha*curr + (1-alpha)*prev

# =========================
# === Plant 학습 함수 ===
# =========================
def train_plant(model, optimizer, criterion, train_loader, val_loader, ckpt_dir, plant):
    ckpt_last = os.path.join(ckpt_dir,f"{plant}_last.pt")
    ckpt_best = os.path.join(ckpt_dir,f"{plant}_best.pt")
    start_epoch, best_r2 = 1, -np.inf

    # Resume 기능
    if RESUME:
        if os.path.exists(ckpt_best):
            start_epoch, best_r2 = load_checkpoint(model, optimizer, ckpt_best)
            print(f"🔄 Resume from best ckpt: {ckpt_best} | start_epoch={start_epoch} | best_r2={best_r2:.4f}")
        elif os.path.exists(ckpt_last):
            start_epoch, best_r2 = load_checkpoint(model, optimizer, ckpt_last)
            print(f"🔄 Resume from last ckpt: {ckpt_last} | start_epoch={start_epoch} | best_r2={best_r2:.4f}")

    r2_ema, early_counter = None, 0
    epoch_start_time = time.time()
    for epoch in range(start_epoch, EPOCHS+1):
        # Train
        model.train()
        train_losses = []
        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
        train_loss_mean = np.mean(train_losses) if train_losses else float("nan")

        # Validation
        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad():
            for Xv, yv in val_loader:
                Xv, yv = Xv.to(DEVICE), yv.to(DEVICE)
                yp = model(Xv)
                val_preds.append(yp[:,0].cpu().numpy())
                val_true.append(yv[:,0].cpu().numpy())
                
        val_preds = np.concatenate(val_preds) if val_preds else np.array([])
        val_true = np.concatenate(val_true) if val_true else np.array([])

        # ✅ 정합성 보정
        if len(val_preds) != len(val_true):
            min_len = min(len(val_preds), len(val_true))
            val_preds, val_true = val_preds[:min_len], val_true[:min_len]

        val_r2, val_rmse, val_mae = calc_metrics(val_true, val_preds)
        r2_ema = ema_update(r2_ema, val_r2)

        best_mark=""
        if r2_ema > best_r2:
            best_r2 = r2_ema
            save_checkpoint(ckpt_best, model, optimizer, epoch, best_r2)
            best_mark="★Best★"
            early_counter=0
        else:
            early_counter+=1

        save_checkpoint(ckpt_last, model, optimizer, epoch, best_r2)

        # ETA
        avg_epoch_time = (time.time()-epoch_start_time)/max(1,epoch-start_epoch+1)
        eta_s = avg_epoch_time*(EPOCHS-epoch)
        eta_str=f"{eta_s:.1f}s" if eta_s<60 else f"{eta_s/60:.2f}m"
        print(f"Epoch {epoch}/{EPOCHS} | TrainLoss={train_loss_mean:.6f} | Val R2={val_r2:.4f} | EMA R2={r2_ema:.4f} {best_mark} | RMSE={val_rmse:.4f} | MAE={val_mae:.4f} | ETA={eta_str}")

        if early_counter>=PATIENCE:
            print(f"⏹ EarlyStopping Triggered for {plant}")
            break

    # 최종 평가
    ckpt_eval = ckpt_best if os.path.exists(ckpt_best) else ckpt_last
    ck = torch.load(ckpt_eval,map_location=DEVICE)
    model.load_state_dict(ck["model_state"])
    val_preds_list, val_true_list = [], []
    with torch.no_grad():
        for Xv, yv in val_loader:
            Xv, yv = Xv.to(DEVICE), yv.to(DEVICE)
            yp = model(Xv)
            val_preds_list.append(yp[:,0].cpu().numpy())
            val_true_list.append(yv[:,0].cpu().numpy())
    val_preds_final = np.concatenate(val_preds_list)
    val_true_final = np.concatenate(val_true_list)

    # ✅ 정합성 보정
    if len(val_preds_final) != len(val_true_final):
        min_len = min(len(val_preds_final), len(val_true_final))
        val_preds_final, val_true_final = val_preds_final[:min_len], val_true_final[:min_len]

    v_r2,v_rmse,v_mae = calc_metrics(val_true_final,val_preds_final)
    return ckpt_eval, (v_r2,v_rmse,v_mae)

# =========================
# === Plant 학습 + 2차 Weight Step ===
# =========================
results_plant_1st, results_plant_2nd = {}, {}
plant_models_1st, plant_models_2nd = {}, {}
plant_summary_rows = []  # HTML 테이블용

for plant, df_train in train_raw.groupby(GROUP_COL):
    print(f"\n🌱 Plant 1차 학습 시작: {plant} | train size: {len(df_train)}")
    df_val = val_raw[val_raw[GROUP_COL]==plant]
    if len(df_val)==0: continue

    train_loader = DataLoader(TimeSeriesSeqDataset(df_train), batch_size=BATCH, shuffle=True)
    val_loader = DataLoader(TimeSeriesSeqDataset(df_val), batch_size=BATCH, shuffle=False)
    model = CNN_BiLSTM().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.MSELoss()
    plant_dir = os.path.join(CKPT_DIR,"plant",plant)
    os.makedirs(plant_dir, exist_ok=True)

    # 1차 학습
    ckpt_1st, metrics_1st = train_plant(model, optimizer, criterion, train_loader, val_loader, plant_dir, plant)
    plant_models_1st[plant] = ckpt_1st
    results_plant_1st[plant] = metrics_1st
    print(f"✅ Plant 1차 평가 완료 | R2={metrics_1st[0]:.4f} | RMSE={metrics_1st[1]:.4f} | MAE={metrics_1st[2]:.4f}")

    # =========================
    # === Plant별 summary row 추가
    # =========================
    r2, rmse, mae = results_plant_1st[plant]
    plant_summary_rows.append({
        "plant": plant,
        "n_train": len(df_train),
        "r2": r2,
        "rmse": rmse,
        "mae": mae
    })

# =========================
# === (NEW) 1차 전체 평가 ===
# =========================
weights_1st = np.array([len(train_raw[train_raw[GROUP_COL]==p]) for p in results_plant_1st.keys()])
r2_1st_all = np.array([results_plant_1st[p][0] for p in results_plant_1st.keys()])
rmse_1st_all = np.array([results_plant_1st[p][1] for p in results_plant_1st.keys()])
mae_1st_all = np.array([results_plant_1st[p][2] for p in results_plant_1st.keys()])
weighted_r2_1st = (r2_1st_all * weights_1st).sum() / weights_1st.sum()
weighted_rmse_1st = (rmse_1st_all * weights_1st).sum() / weights_1st.sum()
weighted_mae_1st = (mae_1st_all * weights_1st).sum() / weights_1st.sum()
print(f"\n📊 [1차 전체평가] 가중평균 R²={weighted_r2_1st:.4f} | RMSE={weighted_rmse_1st:.4f} | MAE={weighted_mae_1st:.4f}")

# 발전소 → 지역 매핑
plant_to_region = train_raw.set_index(GROUP_COL)[REGION_COL].to_dict()

# =========================
# === Region Fine-tuning ===
# =========================
region_models, region_metrics = {}, {}
region_summary_rows = []
region_to_plants = []

print("\n🌿 Region Fine-tuning 시작")

# Region별 Fine-tuning
for region, df_region in train_raw.groupby(REGION_COL):
    df_val_region = val_raw[val_raw[REGION_COL]==region]
    if len(df_val_region)==0: continue

    region_dir = os.path.join(CKPT_DIR,"region",region)
    os.makedirs(region_dir, exist_ok=True)

    plants_in_region = df_region[GROUP_COL].unique().tolist()
    plant_ckpts = [plant_models_2nd[p] for p in plants_in_region if p in plant_models_2nd]
    model = CNN_BiLSTM().to(DEVICE)

    # Plant ckpt 평균으로 초기화
    if plant_ckpts:
        ck_states = [torch.load(ck,map_location=DEVICE)["model_state"] for ck in plant_ckpts]
        avg_state = {k: torch.stack([s[k] for s in ck_states],0).mean(0) for k in ck_states[0].keys()}
        model.load_state_dict(avg_state)
        print(f"🔹 Region {region} 초기화 완료 (Plant 평균 ckpt)")

    optimizer = optim.Adam(model.parameters(), lr=LR*0.5)
    criterion = nn.MSELoss()
    train_loader = DataLoader(TimeSeriesSeqDataset(df_region), batch_size=BATCH, shuffle=True)
    val_loader = DataLoader(TimeSeriesSeqDataset(df_val_region), batch_size=BATCH, shuffle=False)

    best_r2 = -np.inf
    best_ckpt = os.path.join(region_dir,f"{region}_best.pt")
    start_epoch = 1

    # Resume 체크
    if RESUME and os.path.exists(best_ckpt):
        start_epoch, best_r2 = load_checkpoint(model, optimizer, best_ckpt)
        print(f"🔄 Resume Region ckpt: {region} | start_epoch={start_epoch} | best_r2={best_r2:.4f}")

    for epoch in range(start_epoch, FINE_TUNE_EPOCHS+1):
        model.train()
        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()

        # Validation
        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad():
            for Xv, yv in val_loader:
                Xv, yv = Xv.to(DEVICE), yv.to(DEVICE)
                yp = model(Xv)
                val_preds.append(yp[:,0].cpu().numpy())
                val_true.append(yv[:,0].cpu().numpy())
        val_preds_final = np.concatenate(val_preds)
        val_true_final = np.concatenate(val_true)

        # ✅ 정합성 보정
        if len(val_preds_final) != len(val_true_final):
            min_len = min(len(val_preds_final), len(val_true_final))
            val_preds_final, val_true_final = val_preds_final[:min_len], val_true_final[:min_len]

        r2, rmse, mae = calc_metrics(val_true_final, val_preds_final)

        if r2 > best_r2:
            best_r2 = r2
            save_checkpoint(best_ckpt, model, optimizer, epoch, best_r2)
    region_models[region] = best_ckpt
    region_metrics[region] = (best_r2, rmse, mae)
    print(f"✅ Region {region} 최종 평가 | R2={best_r2:.4f} | RMSE={rmse:.4f} | MAE={mae:.4f}")

    # =========================
    # === Region summary row 추가
    # =========================
    r2, rmse, mae = region_metrics[region]
    region_summary_rows.append({
        "region": region,
        "n_train": len(df_region),
        "r2": r2,
        "rmse": rmse,
        "mae": mae
})

# =========================
# === (NEW) 2차 전체평가 ===
# =========================
weights_2nd = np.array([len(train_raw[train_raw[REGION_COL]==p]) for p in region_metrics.keys()])
r2_2nd_all = np.array([region_metrics[p][0] for p in region_metrics.keys()])
rmse_2nd_all = np.array([region_metrics[p][1] for p in region_metrics.keys()])
mae_2nd_all = np.array([region_metrics[p][2] for p in region_metrics.keys()])
weighted_r2_2nd = (r2_2nd_all * weights_2nd).sum() / weights_2nd.sum()
weighted_rmse_2nd = (rmse_2nd_all * weights_2nd).sum() / weights_2nd.sum()
weighted_mae_2nd = (mae_2nd_all * weights_2nd).sum() / weights_2nd.sum()
print(f"\n📊 [2차 전체평가] 가중평균 R²={weighted_r2_2nd:.4f} | RMSE={weighted_rmse_2nd:.4f} | MAE={weighted_mae_2nd:.4f}")

region_to_plants = (
    train_raw.groupby(REGION_COL)[GROUP_COL]
    .unique()
    .apply(list)
    .to_dict()
)

# =========================
# Permutation Importance 기반 feature 영향도 분석
# =========================
def analyze_permutation_importance(region, ckpt_path, test_df):
    region_perm_images_local = {}
    perm_top3_features_local = {}  # top3 feature만 별도 저장
    try:
        ck = torch.load(ckpt_path, map_location=DEVICE)
        st = ck.get("model_state", ck)
        model.load_state_dict(st)
        model.to(DEVICE)
        model.eval()

        df_region = test_df[test_df[REGION_COL] == region].reset_index(drop=True)
        if len(df_region) == 0:
            region_perm_images_local[region] = None
            perm_top3_features_local[region] = []
            return region_perm_images_local, perm_top3_features_local

        ds_train = TimeSeriesSeqDataset(df_region)
        n_windows = len(ds_train)
        if n_windows == 0:
            region_perm_images_local[region] = None
            perm_top3_features_local[region] = []
            return region_perm_images_local, perm_top3_features_local

        # 샘플링
        sample_size = min(PERM_SAMPLE, n_windows)
        X_sample = torch.stack([ds_train[i][0] for i in range(sample_size)]).to(DEVICE)
        y_raw = torch.stack([ds_train[i][1] for i in range(sample_size)]).cpu().numpy()
        with torch.no_grad():
            y_pred_all = model(X_sample).detach().cpu().numpy()

        y_true_col = (y_raw[:, 0] if y_raw.ndim > 1 else y_raw).astype(float)
        y_pred_col = y_pred_all[:, 0].astype(float)
        min_len = min(len(y_true_col), len(y_pred_col))
        y_true_pi = y_true_col[:min_len]
        y_pred_pi = y_pred_col[:min_len]
        base_rmse = np.sqrt(mean_squared_error(y_true_pi, y_pred_pi))

        # Permutation Importance
        X_np = X_sample.detach().cpu().numpy().astype(np.float32)
        seq_len = X_np.shape[1]
        weather_idx = [i for i, f in enumerate(feature_cols) if f in WEATHER_COLS]
        if len(weather_idx) == 0:
            region_perm_images_local[region] = None
            perm_top3_features_local[region] = []
            return region_perm_images_local, perm_top3_features_local

        pi_scores = np.zeros(len(weather_idx))
        for wi, f_idx in enumerate(weather_idx):
            X_perm = X_np.copy()
            for t in range(seq_len):
                np.random.shuffle(X_perm[:, t, f_idx])
            with torch.no_grad():
                y_perm = model(torch.tensor(X_perm, dtype=torch.float32, device=DEVICE)).detach().cpu().numpy()
            y_perm_col = y_perm[:min_len, 0]
            pi_scores[wi] = np.sqrt(mean_squared_error(y_true_pi, y_perm_col)) - base_rmse

        top_idx = np.argsort(pi_scores)[::-1][:3]
        top_features = [feature_cols[weather_idx[i]] for i in top_idx]
        perm_top3_features_local[region] = top_features

        # 시각화
        plt.figure(figsize=(6,3))
        labels = [feature_cols[weather_idx[i]] for i in top_idx[::-1]]
        vals = pi_scores[top_idx][::-1]
        plt.barh(range(len(vals)), vals, height=0.6, color="#3498db")
        plt.yticks(range(len(vals)), labels)
        plt.xlabel("RMSE 증가량")
        plt.title(f"[{region}] Top{len(vals)} Weather Feature Importance")
        buf = io.BytesIO()
        plt.tight_layout()
        plt.savefig(buf, format="png", bbox_inches="tight")
        buf.seek(0)
        region_perm_images_local[region] = base64.b64encode(buf.read()).decode("utf-8")
        buf.close()
        plt.close()

    except Exception as e:
        print(f"⚠ Permutation Importance 실패 ({region}): {e}")
        region_perm_images_local[region] = None
        perm_top3_features_local[region] = []

    return region_perm_images_local, perm_top3_features_local

def detect_outliers(region_data, region_scalers=None, SEQ_LEN=24, batch_size=64):
    import matplotlib.ticker as mtick
    import io, base64
    actual_vs_pred_images_local = {}
    outlier_reports_local = {}

    try:
        all_y_true_list, all_y_pred_list, all_dates_list = [], [], []
        region_preds = {}

        for region, info in region_data.items():
            try:
                df_region = info["df"].reset_index(drop=True)
                ckpt_path = info["ckpt_path"]

                if len(df_region) <= SEQ_LEN:
                    outlier_reports_local[region] = {"count":0, "ratio":0.0, "threshold":0.0}
                    continue

                # --- 모델 로드 ---
                ck = torch.load(ckpt_path, map_location=DEVICE)
                st = ck.get("model_state", ck)
                model.load_state_dict(st)
                model.to(DEVICE)
                model.eval()

                # --- 테스트 시퀀스 생성 ---
                ds_test = TimeSeriesSeqDataset(df_region, seq_len=SEQ_LEN)
                n_windows = len(ds_test)
                if n_windows == 0:
                    outlier_reports_local[region] = {"count":0, "ratio":0.0, "threshold":0.0}
                    continue

                X_list, y_list, dates_list = [], [], []
                for i in range(n_windows):
                    x_i, y_i = ds_test[i]
                    X_list.append(np.asarray(x_i))
                    y_list.append(np.asarray(y_i))
                    if i + SEQ_LEN >= len(df_region):
                        continue
                    dates_list.append(pd.to_datetime(df_region.loc[i + SEQ_LEN, TIME_COL]))

                if not dates_list:
                    outlier_reports_local[region] = {"count":0, "ratio":0.0, "threshold":0.0}
                    continue

                X_np = np.stack(X_list)
                y_raw = np.stack(y_list)

                # --- 예측 ---
                y_pred_list = []
                with torch.no_grad():
                    for start_idx in range(0, len(X_np), batch_size):
                        end_idx = start_idx + batch_size
                        X_batch = torch.tensor(X_np[start_idx:end_idx], dtype=torch.float32, device=DEVICE)
                        y_pred_list.append(model(X_batch).detach().cpu().numpy())
                y_pred_all = np.vstack(y_pred_list)

                # --- 길이 맞추기 ---
                y_pred_col = y_pred_all[:,0].astype(float)
                y_true_col = (y_raw[:,0] if y_raw.ndim>1 else y_raw).astype(float)
                min_len = min(len(y_true_col), len(y_pred_col), len(dates_list))
                y_true_col = y_true_col[:min_len]
                y_pred_col = y_pred_col[:min_len]
                target_dates = np.array(dates_list[:min_len])

                # --- 역변환 (MWh) ---
                if region_scalers and region in region_scalers:
                    _, _, tsc = region_scalers[region]
                    time_step_hours = 24

                    y_true_2d = y_true_col.reshape(-1,1)
                    y_pred_2d = y_pred_col.reshape(-1,1)

                    y_true_inv = tsc.inverse_transform(y_true_2d)
                    y_pred_inv = tsc.inverse_transform(y_pred_2d)

                    y_true_inv = np.clip(np.expm1(y_true_inv), 0, None)
                    y_pred_inv = np.clip(np.expm1(y_pred_inv), 0, None)

                    y_true_mwh = y_true_inv.flatten() * time_step_hours
                    y_pred_mwh = y_pred_inv.flatten() * time_step_hours
                else:
                    # 스케일러 없으면 단순 MW → MWh 변환
                    time_step_hours = 24
                    y_true_mwh = np.clip(y_true_col, 0, None) * time_step_hours
                    y_pred_mwh = np.clip(y_pred_col, 0, None) * time_step_hours

                # --- 이상치 탐지 ---
                residuals = np.abs(y_true_mwh - y_pred_mwh)
                threshold = np.quantile(residuals, 0.99)
                outliers_mask = residuals > threshold
                outlier_reports_local[region] = {
                    "count": int(outliers_mask.sum()),
                    "ratio": float(outliers_mask.mean()),
                    "threshold": float(threshold)
                }

                all_y_true_list.append(y_true_mwh)
                all_y_pred_list.append(y_pred_mwh)
                all_dates_list.append(target_dates)
                region_preds[region] = (y_true_mwh, y_pred_mwh, target_dates, outliers_mask, threshold)

            except Exception as e:
                print(f"⚠ 지역 처리 실패: {region} ({e})")
                outlier_reports_local[region] = {"count":0, "ratio":0.0, "threshold":0.0}

        # =========================
        # 전체 그래프
        # =========================
        if all_y_true_list:
            all_y_true = np.concatenate(all_y_true_list)
            all_y_pred = np.concatenate(all_y_pred_list)
            all_dates = np.concatenate(all_dates_list)
            
            residuals_all = np.abs(all_y_true - all_y_pred)
            threshold_all = np.quantile(residuals_all, 0.99)
            outliers_mask_all = residuals_all > threshold_all
            outlier_count_all = int(outliers_mask_all.sum())
            outlier_ratio_all = float(outlier_count_all / max(1, len(residuals_all)))
            
            plt.figure(figsize=(14, 5))
            plt.plot(all_dates, all_y_true, color='blue', label='실제값', alpha=0.8, linewidth=1.5, zorder=1)
            plt.plot(all_dates, all_y_pred, color='orange', label='예측값', alpha=0.7, linewidth=1.5, zorder=2)
            if outlier_count_all > 0:
                plt.scatter(
                    all_dates[outliers_mask_all],
                    all_y_true[outliers_mask_all],
                    color='red', s=15,
                    label=f'이상치 ({outlier_count_all}건, {outlier_ratio_all:.2%})',
                    zorder=3
                )

            plt.title(f"[전체 지역] 예측 vs 실제", fontsize=13, weight='bold')
            plt.xlabel("날짜", fontsize=11)
            plt.ylabel("합산 발전량 (MWh)", fontsize=11)
            plt.legend(loc='upper right')
            plt.grid(True, alpha=0.3)
            # 🔽 y축 통일 포맷 적용
            ax = plt.gca()
            ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, pos: f"{x:.0f}"))
            plt.tight_layout()
            plt.ylim(bottom=0)
            buf = io.BytesIO()
            plt.savefig(buf, format="png", bbox_inches="tight")
            buf.seek(0)
            actual_vs_pred_images_local["전체"] = base64.b64encode(buf.read()).decode("utf-8")
            buf.close()
            plt.close()
            # 전체 이상치 정보도 별도 dict로 저장
            outlier_reports_local["전체"] = {
                "count": outlier_count_all,
                "ratio": outlier_ratio_all,
                "threshold": threshold_all
            }

        # =========================
        # 지역별 그래프
        # =========================
        for region, vals in region_preds.items():
            y_true, y_pred, dates, outliers_mask, threshold = vals
            outlier_count = int(outliers_mask.sum())
            outlier_ratio = float(outlier_count / max(1, len(y_true)))

            plt.figure(figsize=(12, 4))
            plt.plot(dates, y_true, color='blue', label='실제값', alpha=0.8, linewidth=1.5, zorder=1)
            plt.plot(dates, y_pred, color='orange', label='예측값', alpha=0.7, linewidth=1.5, zorder=2)
            if outlier_count > 0:
                plt.scatter(
                    dates[outliers_mask],
                    y_true[outliers_mask],
                    color='red', s=15,
                    label=f'이상치 ({outlier_count}건, {outlier_ratio:.2%})',
                    zorder=3
                )

            plt.title(f"[{region}] 예측 vs 실제", fontsize=12, weight='bold')
            plt.xlabel("날짜", fontsize=11)
            plt.ylabel("합산 발전량 (MWh)", fontsize=11)
            plt.legend(loc='upper right')
            plt.grid(True, alpha=0.3)
            # 🔽 y축 통일 포맷 적용
            ax = plt.gca()
            ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, pos: f"{x:.0f}"))
            plt.tight_layout()
            plt.ylim(bottom=0)
            buf = io.BytesIO()
            plt.savefig(buf, format="png", bbox_inches="tight")
            buf.seek(0)
            actual_vs_pred_images_local[region] = base64.b64encode(buf.read()).decode("utf-8")
            buf.close()
            plt.close()

        return actual_vs_pred_images_local, outlier_reports_local

    except Exception as e:
        print(f"⚠ 이상치 탐지 실패: {e}")
        return actual_vs_pred_images_local, outlier_reports_local

# =========================
# === 전체 실행
# =========================
perm_images_all = {}
perm_top3_features_all = {}

for region, ckpt in region_models.items():
    perm_images, top3_features = analyze_permutation_importance(region, ckpt, test_raw)
    perm_images_all.update(perm_images)
    perm_top3_features_all.update(top3_features)

region_data = {}
for region, ckpt_path in region_models.items():
    df_region = test_raw[test_raw[REGION_COL] == region].reset_index(drop=True)
    scaler = region_scalers.get(region, None)
    region_data[region] = {
        "df": df_region,
        "ckpt_path": ckpt_path,
        "scaler": scaler,
    }

pred_images, outlier_reports_local = detect_outliers(region_data, SEQ_LEN=24, batch_size=64)

# =========================
# === HTML 리포트 생성
# =========================

html_template = """
<html lang="ko">
<head>
<meta charset="utf-8"/>
<title>📊 발전소·지역별 성능 및 이상치 분석 리포트</title>
<style>
body { font-family: 'Pretendard', sans-serif; margin: 30px; background-color:#fafafa; color:#222; }
h1, h2 { margin-top: 40px; color:#333; }
table { border-collapse: collapse; width: 100%; margin: 20px 0; font-size: 13px; }
th, td { border: 1px solid #ccc; padding: 8px 12px; text-align: center; }
th { background-color: #d4edda; font-weight: bold; color:#111; }
tbody tr:nth-child(odd) { background-color: #fcfcfc; }
tbody tr:hover { background-color: #f5f9ff; }
.perm-img { width: 240px; border-radius: 10px; box-shadow: 0 2px 6px rgba(0,0,0,0.12); }
.small { font-size: 12px; color: #666; }
.card { background:white; padding:15px; border-radius:12px; box-shadow:0 3px 8px rgba(0,0,0,0.08); margin-top:20px; }
img { max-width:100%; height:auto; border-radius:8px; }
.index-col { width:40px; font-weight: bold; }
.plant-col { width:180px; text-align:left; padding-left:10px; }
.region-col { width:120px; text-align:left; padding-left:10px; }
.metric-col { width:80px; }
.graph-col { width:260px; }
</style>
</head>
<body>

<h1>📈 발전소별 성능 요약</h1>
<table>
<thead>
<tr>
<th class="index-col"></th>
<th class="plant-col">발전소</th>
<th class="region-col">지역</th>
<th class="metric-col">표본 수</th>
<th class="metric-col">R²</th>
<th class="metric-col">RMSE</th>
<th class="metric-col">MAE</th>
</tr>
</thead>
<tbody>
{% for row in plant_rows %}
<tr>
<td class="index-col">{{ loop.index }}</td>
<td class="plant-col">{{ row.plant }}</td>
<td class="region-col">{{ plant_to_region.get(row.plant, '—') }}</td>
<td class="metric-col">{{ row.n_train if row.n_train is defined else '—' }}</td>
<td class="metric-col">{{ "%.4f"|format(row.r2) if row.r2 is defined else '—' }}</td>
<td class="metric-col">{{ "%.4f"|format(row.rmse) if row.rmse is defined else '—' }}</td>
<td class="metric-col">{{ "%.4f"|format(row.mae) if row.mae is defined else '—' }}</td>
</tr>
{% endfor %}
</tbody>
</table>

<h1>🌍 지역별 성능 및 Permutation Importance</h1>
<table>
<thead>
<tr>
<th class="index-col"></th>
<th class="region-col">지역</th>
<th class="plant-col">발전소</th>
<th class="metric-col">표본 수</th>
<th class="metric-col">R²</th>
<th class="metric-col">RMSE</th>
<th class="metric-col">MAE</th>
<th class="metric-col">Top3 영향 요인</th>
<th class="graph-col">Permutation Importance</th>
</tr>
</thead>
<tbody>
{% for row in region_rows %}
<tr>
<td class="index-col">{{ loop.index }}</td>
<td class="region-col">{{ row.region }}</td>
<td class="plant-col">
{% if region_to_plants and region_to_plants.get(row.region) %}
{{ ', '.join(region_to_plants[row.region]) }}
{% else %}<span class="small">—</span>{% endif %}
</td>
<td class="metric-col">{{ row.n_train if row.n_train is defined else '—' }}</td>
<td class="metric-col">{{ "%.4f"|format(row.r2) if row.r2 is defined else '—' }}</td>
<td class="metric-col">{{ "%.4f"|format(row.rmse) if row.rmse is defined else '—' }}</td>
<td class="metric-col">{{ "%.4f"|format(row.mae) if row.mae is defined else '—' }}</td>
<td class="metric-col">
{% if perm_top3_features and perm_top3_features.get(row.region) %}
{{ ', '.join(perm_top3_features[row.region]) }}
{% else %} — {% endif %}
</td>
<td class="graph-col">
{% if perm_images and perm_images.get(row.region) %}
<img class="perm-img" src="data:image/png;base64,{{ perm_images[row.region] }}" alt="pi-{{row.region}}"/>
{% else %}<span class="small">N/A</span>{% endif %}
</td>
</tr>
{% endfor %}
</tbody>
</table>

<div class="card">
<h2>📘 전체 가중 평균 성능 요약</h2>
<p class="middle">
1차 학습 가중 평균 → R²={{"%.4f"|format(weighted_r2_1st)}} | RMSE={{"%.4f"|format(weighted_rmse_1st)}} | MAE={{"%.4f"|format(weighted_mae_1st)}}<br>
2차 학습 가중 평균 → R²={{"%.4f"|format(weighted_r2_2nd)}} | RMSE={{"%.4f"|format(weighted_rmse_2nd)}} | MAE={{"%.4f"|format(weighted_mae_2nd)}}
</p>
</div>

<h1>📊 지역별 예측 결과 및 이상치</h1>

<!-- 전체 그래프 -->
<div class="card">
  <h2>전체</h2>
  {% if outlier_reports and outlier_reports.get("전체") %}
  <p class="small">
    이상치 개수: {{ outlier_reports["전체"]['count'] }} |
    비율: {{"%.2f"|format(outlier_reports["전체"]['ratio']*100)}}% |
    임계값: {{"%.4f"|format(outlier_reports["전체"]['threshold'])}}Mwh
  </p>
  {% endif %}
  <img src="data:image/png;base64,{{ pred_images["전체"] }}" alt="actual-pred-전체">
</div>

<!-- 지역별 그래프 반복 -->
{% for region, img_b64 in pred_images.items() %}
  {% if region != "전체" %}
  <div class="card">
    <h2>{{ region }}</h2>
    {% if outlier_reports and outlier_reports.get(region) %}
    <p class="small">
      이상치 개수: {{ outlier_reports[region]['count'] }} |
      비율: {{"%.2f"|format(outlier_reports[region]['ratio']*100)}}% |
      임계값: {{"%.4f"|format(outlier_reports[region]['threshold'])}}Mwh
    </p>
    {% endif %}
    <img src="data:image/png;base64,{{ img_b64 }}" alt="actual-pred-{{region}}">
  </div>
  {% endif %}
{% endfor %}

</body>
</html>
"""

# =========================
# HTML 렌더링
# =========================
from jinja2 import Template

template = Template(html_template)
html_out = template.render(
    plant_rows=plant_summary_rows,
    region_rows=region_summary_rows,
    region_to_plants=region_to_plants,
    plant_to_region=plant_to_region,
    perm_images=perm_images_all,
    perm_top3_features=perm_top3_features_all,
    outlier_reports=outlier_reports_local,
    pred_images=pred_images,
    weighted_r2_1st=weighted_r2_1st,
    weighted_rmse_1st=weighted_rmse_1st,
    weighted_mae_1st=weighted_mae_1st,
    weighted_r2_2nd=weighted_r2_2nd,
    weighted_rmse_2nd=weighted_rmse_2nd,
    weighted_mae_2nd=weighted_mae_2nd,
)

with open("region_outlier_perm_report.html", "w", encoding="utf-8") as f:
    f.write(html_out)

html_file = os.path.join(SAVE_DIR,"plant_region_report_perm.html")
with open(html_file,"w",encoding="utf-8") as f:
    f.write(html_out)
webbrowser.open(f"file://{html_file}")
print(f"✅ HTML 보고서 생성 완료 (perm 포함): {html_file}")


✅ Device: cuda
✅ feature_cols (22): ['기온(°C)', '강수량(mm)', '풍속(m/s)', '습도(%)', '증기압(hPa)', '일조(hr)', '일사(MJ/m2)', '적설(cm)', '전운량(10분위)', '중하층운량(10분위)', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos', 'lag_1', 'lag_3', 'lag_6', 'lag_24', 'diff_1', 'diff_3', 'diff_6', 'diff_24']

🌱 Plant 1차 학습 시작: 남제주소내 | train size: 78888
🔄 Resume from best ckpt: C:\ESG_Project1\cnn_bilstm\output\checkpoints\plant\남제주소내\남제주소내_best.pt | start_epoch=8 | best_r2=0.8989
Epoch 8/50 | TrainLoss=0.120836 | Val R2=0.8957 | EMA R2=0.8957  | RMSE=0.2855 | MAE=0.1578 | ETA=11.11m
Epoch 9/50 | TrainLoss=0.104145 | Val R2=0.8658 | EMA R2=0.8867  | RMSE=0.3239 | MAE=0.1846 | ETA=10.71m
Epoch 10/50 | TrainLoss=0.089934 | Val R2=0.8434 | EMA R2=0.8737  | RMSE=0.3499 | MAE=0.2005 | ETA=10.41m
Epoch 11/50 | TrainLoss=0.078714 | Val R2=0.8220 | EMA R2=0.8582  | RMSE=0.3731 | MAE=0.2050 | ETA=10.16m
Epoch 12/50 | TrainLoss=0.068393 | Val R2=0.8241 | EMA R2=0.8480  | RMSE=0.3708 | MAE=0.1953 | ETA=9.87m
Epoch 13/50 | TrainLoss